# NB-P01-01 — Python Fundamentals

Companion notebook for P01 lessons 01–05. Run every cell with **Shift+Enter**, top to bottom.

Nothing here needs a network connection or a file you did not create: every example is self-contained,
and the data is generated with a fixed seed where randomness is involved.

**How to use it:** read the markdown, predict the output of each code cell, then run it. A prediction
that is wrong is more useful than one that is right.

## 1. Check your environment

In [1]:
import platform
import sys

print("Python", platform.python_version())
print("Executable:", sys.executable)

import numpy as np
import pandas as pd

print("NumPy", np.__version__, "| pandas", pd.__version__)

Python 3.14.7
Executable: C:\Python314\python.exe


NumPy 2.5.3 | pandas 3.0.6


If the versions fail to import, your notebook is running a different interpreter from the one you
installed into. Select the right kernel before continuing.

## 2. Variables, types and operators

Python has five types you will use constantly: `int`, `float`, `str`, `bool`, `None`. A name is bound to a
value; the value carries the type.

In [2]:
age = 21
name = "Ayesha"
passed = age >= 18
nothing = None

for value in (age, name, passed, nothing):
    print(f"{value!r:>12}  ->  {type(value).__name__}")

          21  ->  int
    'Ayesha'  ->  str
        True  ->  bool
        None  ->  NoneType


Two operators produce results beginners do not expect: `//` floors and `%` gives the remainder. They
are two halves of one identity, $a = b(a // b) + (a \bmod b)$.

In [3]:
print("17 // 5 =", 17 // 5)
print("17 %  5 =", 17 % 5)
print("137 minutes =", divmod(137, 60), "(hours, minutes)")

celsius = 37
print("37 C =", round((9 / 5) * celsius + 32, 2), "F")

17 // 5 = 3
17 %  5 = 2
137 minutes = (2, 17) (hours, minutes)
37 C = 98.6 F


### Floating point is not exact

`0.1 + 0.2` is not `0.3` in binary floating point. Compare with a tolerance, never with `==`.

In [4]:
import math

print(0.1 + 0.2)
print(0.1 + 0.2 == 0.3)
print(math.isclose(0.1 + 0.2, 0.3))

0.30000000000000004
False
True


## 3. Conditionals and loops

Only the first matching branch runs, so the order of conditions is part of the logic.

In [5]:
def classify(n: int) -> str:
    """Return a label for an integer."""
    if n < 0:
        return "negative"
    if n == 0:
        return "zero"
    if n % 2 == 0:
        return "positive even"
    return "positive odd"


print([classify(n) for n in (-4, 0, 3, 8)])

['negative', 'zero', 'positive odd', 'positive even']


In [6]:
for n in range(1, 16):
    if n % 15 == 0:
        print("FizzBuzz")
    elif n % 3 == 0:
        print("Fizz")
    elif n % 5 == 0:
        print("Buzz")
    else:
        print(n)

1
2
Fizz
4
Buzz
Fizz
7
8
Fizz
Buzz
11
Fizz
13
14
FizzBuzz


The `% 15` test must come first: 15 is divisible by both 3 and 5, and the first match wins.

## 4. Comprehensions and lambda

A comprehension is set-builder notation with executable syntax.

In [7]:
squares = [x * x for x in range(1, 11)]
even_squares = [x * x for x in range(1, 11) if x % 2 == 0]
lengths = {word: len(word) for word in ["ai", "agent", "search"]}
firsts = {word[0] for word in ["ai", "agent", "search"]}

print(squares)
print(even_squares)
print(lengths)
print(firsts)

[1, 4, 9, 16, 25, 36, 49, 64, 81, 100]
[4, 16, 36, 64, 100]
{'ai': 2, 'agent': 5, 'search': 6}
{'a', 's'}


In [8]:
people = [("Sara", 31), ("Ali", 24), ("Hina", 28)]
print(sorted(people, key=lambda p: p[1]))
print(sorted(people, key=lambda p: -p[1])[:2])

[('Ali', 24), ('Hina', 28), ('Sara', 31)]
[('Sara', 31), ('Hina', 28)]


## 5. Functions and collections

The four collections answer different questions: sequence (`list`), fixed record (`tuple`), uniqueness
(`set`), lookup (`dict`).

Write functions that **return**; printing hides the value from the rest of the program.

In [9]:
def stdev(xs, sample=True):
    """Standard deviation; sample=True divides by n-1."""
    m = sum(xs) / len(xs)
    n = len(xs) - (1 if sample else 0)
    return (sum((x - m) ** 2 for x in xs) / n) ** 0.5


data = [2, 4, 4, 4, 5, 5, 7, 9]
print("mean  =", sum(data) / len(data))
print("sd    =", round(stdev(data), 3))
print("pop sd=", round(stdev(data, sample=False), 3))

mean  = 5.0
sd    = 2.138
pop sd= 2.0


In [10]:
from collections import Counter

records = [
    {"name": "Ali", "city": "Lahore", "score": 78},
    {"name": "Sara", "city": "Karachi", "score": 91},
    {"name": "Bilal", "city": "Lahore", "score": 64},
]

by_city = {}
for r in records:
    by_city.setdefault(r["city"], []).append(r["score"])

print({city: round(sum(s) / len(s), 1) for city, s in by_city.items()})
print(Counter(r["city"] for r in records).most_common())

{'Lahore': 71.0, 'Karachi': 91.0}
[('Lahore', 2), ('Karachi', 1)]


### The mutable default trap

The default value is created **once**, when the function is defined, and shared by every call.

In [11]:
def add_broken(item, bucket=[]):
    bucket.append(item)
    return bucket


print(add_broken(1), add_broken(2))


def add_fixed(item, bucket=None):
    if bucket is None:
        bucket = []
    bucket.append(item)
    return bucket


print(add_fixed(1), add_fixed(2))

[1, 2] [1, 2]
[1] [2]


## 6. Exceptions and files

Catch only what you can handle, and report what you rejected instead of silently dropping it.

In [12]:
def parse_scores(lines):
    good, bad = [], []
    for line in lines:
        try:
            good.append(int(line))
        except ValueError:
            bad.append(line)
    return good, bad


print(parse_scores(["78", "oops", "91", ""]))

([78, 91], ['oops', ''])


In [13]:
import json
import tempfile
from pathlib import Path

records = [{"name": "Ali", "scores": [78, 82]}, {"name": "Sara", "scores": [91, 88]}]

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "students.json"
    path.write_text(json.dumps(records, indent=2), encoding="utf-8")
    loaded = json.loads(path.read_text(encoding="utf-8"))
    print(loaded[0]["name"], sum(loaded[0]["scores"]) / len(loaded[0]["scores"]))
    print("bytes on disk:", path.stat().st_size)

Ali 80.0
bytes on disk: 156


## 7. A small class

State and the operations on it travel together.

In [14]:
class GradeBook:
    """Collect scores and summarise them."""

    def __init__(self):
        self.scores = []

    def add(self, score: float) -> None:
        if not 0 <= score <= 100:
            raise ValueError(f"score out of range: {score}")
        self.scores.append(score)

    def mean(self) -> float:
        if not self.scores:
            raise ValueError("no scores yet")
        return sum(self.scores) / len(self.scores)

    def __repr__(self) -> str:
        return f"GradeBook(n={len(self.scores)})"


book = GradeBook()
for s in (78, 91, 64, 85):
    book.add(s)
print(book, "mean =", round(book.mean(), 2))

try:
    book.add(140)
except ValueError as err:
    print("rejected:", err)

GradeBook(n=4) mean = 79.5
rejected: score out of range: 140


## 8. Exercises

Write your answer in the cell, run it, then compare with the solutions at the end.

**Exercise 1.** Write `total_minutes(hours, minutes)` returning the number of minutes.

**Exercise 2.** Given `values = [4, -2, 9, -5, 0]`, build a list of the positive values doubled, using a
comprehension.

**Exercise 3.** Write `word_lengths(text)` returning a dictionary of word → length for the words in a
string, ignoring punctuation-free duplicates (last occurrence wins).

**Exercise 4.** Write `safe_divide(a, b)` returning `None` instead of raising when `b == 0`.

In [15]:
# Exercise 1
def total_minutes(hours, minutes):
    return hours * 60 + minutes


print(total_minutes(2, 17))

137


In [16]:
# Exercise 2
values = [4, -2, 9, -5, 0]
print([v * 2 for v in values if v > 0])

[8, 18]


In [17]:
# Exercise 3
def word_lengths(text):
    return {word: len(word) for word in text.split()}


print(word_lengths("ai is a field"))

{'ai': 2, 'is': 2, 'a': 1, 'field': 5}


In [18]:
# Exercise 4
def safe_divide(a, b):
    if b == 0:
        return None
    return a / b


print(safe_divide(1, 4), safe_divide(1, 0))

0.25 None


## 9. Check yourself

1. Why is `bool("False")` true?
2. What does `list(range(2, 11, 3))` give?
3. Why does the mutable-default version of `add_broken` accumulate across calls?
4. Which collection would you use for a set of unique student ids, and why?

If any answer is unclear, revisit [LP01-02](../lessons/LP01-02-variables-types-and-operators.md) through
[LP01-05](../lessons/LP01-05-objects-modules-exceptions-and-files.md).